<a href="https://colab.research.google.com/github/NandaniChavan02/TECH-INTERN/blob/main/Movie_Recommendation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Movie Recommendation**

In [3]:
import os
import pandas as pd
from surprise import Dataset, SVD, accuracy
from surprise.model_selection import train_test_split


print("Loading MovieLens dataset...")

data = Dataset.load_builtin("ml-100k")

ratings_data = data.raw_ratings

df = pd.DataFrame(
    ratings_data,
    columns=["user_id", "movie_id", "rating", "timestamp"]
)

print("\nDataset Information")
print("---------------------")
print("Total Ratings :", len(df))
print("Total Users   :", df["user_id"].nunique())
print("Total Movies  :", df["movie_id"].nunique())


trainset, testset = train_test_split(
    data,
    test_size=0.2,
    random_state=42
)


model = SVD()

model.fit(trainset)


predictions = model.test(testset)


print("\nModel Evaluation")
print("------------------")

accuracy.rmse(predictions)
accuracy.mae(predictions)


movie_file_path = os.path.expanduser(
    "~/.surprise_data/ml-100k/ml-100k/u.item"
)

movie_titles = {}

if os.path.exists(movie_file_path):

    with open(
        movie_file_path,
        "r",
        encoding="ISO-8859-1"
    ) as file:

        for line in file:
            parts = line.split("|")
            movie_titles[parts[0]] = parts[1]


target_user = "196"
top_n = 5

print(f"\nTop {top_n} Movie Recommendations for User {target_user}")
print("---------------------------------------------------")


watched_movies = set(
    [rating[1] for rating in ratings_data if rating[0] == target_user]
)

all_movies = set(
    [rating[1] for rating in ratings_data]
)

unwatched_movies = all_movies - watched_movies


recommendations = []

for movie_id in unwatched_movies:

    predicted_rating = model.predict(
        target_user,
        movie_id
    )

    recommendations.append(
        (movie_id, predicted_rating.est)
    )


recommendations.sort(
    key=lambda x: x[1],
    reverse=True
)


for index, (movie_id, score) in enumerate(
    recommendations[:top_n],
    start=1
):

    movie_name = movie_titles.get(
        movie_id,
        "Unknown Movie"
    )

    print(
        f"{index}. {movie_name} "
        f"(Predicted Rating: {score:.2f})"
    )

Loading MovieLens dataset...

Dataset Information
---------------------
Total Ratings : 100000
Total Users   : 943
Total Movies  : 1682

Model Evaluation
------------------
RMSE: 0.9353
MAE:  0.7373

Top 5 Movie Recommendations for User 196
---------------------------------------------------
1. Rear Window (1954) (Predicted Rating: 4.56)
2. Silence of the Lambs, The (1991) (Predicted Rating: 4.55)
3. Raise the Red Lantern (1991) (Predicted Rating: 4.50)
4. Close Shave, A (1995) (Predicted Rating: 4.49)
5. Casablanca (1942) (Predicted Rating: 4.49)
